# 05B PHA CGenFF parameters (CHARMM-GUI website)

**For enzyme systems:** run this notebook, then `06B_cgenff_gromacs_pha_enzyme_in_water.ipynb`. For a quick PHA-in-water run, use 05A and 06A instead.

CHARMM/GROMACS: polymer **CGenFF** via CHARMM-GUI Ligand Reader & Modeler · protein **CHARMM36m** · water/ions **CHARMM TIP3P + SOD/CLA** · engine **GROMACS**.

Use the CHARMM/GROMACS route for protein–polymer systems that should stay in one consistent CHARMM force-field family. Use the Amber/OpenMM route (`05A_amber_gaff2_parameters.ipynb` → `06_optional_gaff2_openmm_pha_enzyme_in_water.ipynb`) for the Amber family (GAFF2 / ff19SB / OPC) in OpenMM.

**The enzyme–PHA production simulations (GK13/ANC45 × P3HO_4/P3HB_4) used this route.** CGenFF assigned all atom types, charges and parameters, so no GAFF2 charges enter them.

This notebook documents the manual CHARMM-GUI steps and provides two read-only checks. It does not automate CGenFF.

## CHARMM-GUI tools

- [CHARMM-GUI](https://www.charmm-gui.org)
- [Ligand Reader & Modeler](https://www.charmm-gui.org/?doc=input/ligandrm): CGenFF atom types, charges and parameters for a non-standard molecule.
- [Solution Builder](https://www.charmm-gui.org/?doc=input/solution): solvated box, ions and engine-specific inputs (GROMACS here).
- [Polymer Builder](https://www.charmm-gui.org/?doc=input/polymer): Choi et al., *J. Chem. Theory Comput.* 2021, [doi:10.1021/acs.jctc.1c00169](https://doi.org/10.1021/acs.jctc.1c00169). PHA monomer support not verified.

## Step 1: PHA parameters with CHARMM-GUI Ligand Reader & Modeler

**Input:** the R-configured SDF exported by `04_export_structures.ipynb`, one per oligomer: `PHB4_R.sdf`, `PHB8_R.sdf`, `PHO4_R.sdf`, `PHO8_R.sdf`, `PHDD4_R.sdf`, `PHDD8_R.sdf`.

**Procedure** (on the CHARMM-GUI website):

1. Open **Ligand Reader & Modeler**.
2. **Upload MOL/MOL2/SDF**: choose the PHA SDF, e.g. `PHB4_R.sdf`.
3. Check the structure shown in Marvin JS: CHARMM-GUI builds the topology from that drawing, so the chirality (all backbone stereocentres R), the bond orders and the explicit hydrogens must be correct.
4. Generate the CGenFF parameters and download the result.

**Output** (folder `lig/` of the download):

| File | Content |
| --- | --- |
| `lig.rtf` | CGenFF topology, with a charge penalty on every atom |
| `lig_g.rtf` | the same topology with charge groups |
| `lig.prm` | CGenFF parameters by analogy, with a penalty on every parameter |
| `lig.log`, `ndihe.str` | CGenFF log and extra dihedrals |

**`lig_g.rtf` and `lig.prm` are what Solution Builder asks for in 06B** ("Upload CHARMM top & par for hetero chain"). Check their penalties and version below before using them.

## Open items

- TODO(Zhiwen): CGenFF version used for the production systems (read it from the `lig.rtf` / `lig.prm` header).
  *Evidence from the PHO4 Ligand Reader download (Apr 2026):* CGenFF program version 4.0 (released June 2024), for CGenFF topology and parameter files version 5.0.
- TODO(Zhiwen): which rtf (`lig.rtf` or `lig_g.rtf`) the GROMACS `LIG.itp` charges came from.
  *Evidence from the PHO4 Ligand Reader download (Apr 2026):* `lig.rtf` and `lig_g.rtf` give identical per-atom charges (99/99 atoms; `lig_g.rtf` only regroups them), and `gromacs/LIG.itp` matches both (net charge 0). Confirm for the production systems.
- TODO(Zhiwen): how the enzyme–polymer starting pose was obtained (docking; the complexes are named `..._pose2_complex.pdb`).
  *Evidence from the Solution Builder downloads:* the GK13–PHO4 builds start from `GK13_PHO4_pose2_complex.pdb` (docking pose 2).
- *Production provenance (checked 2026-09-30):* the GK13_P3HO_4 production input is byte-identical to CHARMM-GUI job **8214536317** (22 June 2026, 137,360 atoms), not to the later rebuild 8221445616 (23 June, 137,363 atoms). Both builds have the same LIG types and charges, and all four LIG stereocentres are R in both.
- *Caution:* the April PHO4 Ligand Reader download (job 7687539766) used for the version and charge evidence above contains the **S** enantiomer (all four stereocentres S in its `drawing_3D.mol` and `ligandrm.pdb`). Its CGenFF version, penalties and per-atom charges are unaffected by chirality, but do not use its coordinates for an R system.

## Quality checklist

- **CGenFF penalties:** below 10 the analogy is fair; 10–50 basic validation is recommended; above 50 the parameters need extensive validation or optimisation (check A below).
- **Stereochemistry:** every backbone stereocentre must be R in the SDF sent to CHARMM-GUI (check B below) and still R after CHARMM-GUI (checked in 06B).
- **`[ defaults ]`** in the GROMACS topology is CHARMM-style: `1 2 yes 1.0 1.0`.
- **Net charge and ion counts** match the intended system (checked in 06B).

### Check A: CGenFF penalties

Set `LIG_DIR` to the `lig/` folder of the download. The cell reads `lig.rtf` and `lig.prm` (and `lig_g.rtf` if present) and prints the maximum parameter penalty and the maximum charge penalty, with the entries that set them.

How to read them (CGenFF guidance): **below 10** the analogy is fair; **10–50** some basic validation is recommended; **above 50** the parameters need extensive validation or optimisation. Example (ANC55 / PHB4): parameter penalty 4.5, charge penalty 3.5, both fair.

In [ ]:
from pathlib import Path
import sys

repo_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "iphasimulator").exists())
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))
from iphasimulator.charmmgui_import import penalty_category, read_cgenff_penalties, read_cgenff_version

LIG_DIR = None  # e.g. Path("/path/to/Anc55_PHB4-XXXXXXXXXX/lig")

if LIG_DIR is None:
    print("LIG_DIR is None: set it to the lig/ folder of a CHARMM-GUI download to check penalties.")
else:
    report = read_cgenff_penalties(LIG_DIR)
    print("Files read:", ", ".join(report.files_read))
    for title, value, entries in (
        ("parameter", report.max_parameter_penalty, report.top_parameter_penalties(3)),
        ("charge", report.max_charge_penalty, report.top_charge_penalties(3)),
    ):
        print(f"Max {title} penalty: {value:g} ({penalty_category(value)})")
        for entry in entries:
            print(f"   {entry.penalty:>6g}  {entry.source}: {entry.label}")

### CGenFF version

The header of `lig.rtf` and `lig.prm` states which CGenFF program made the parameters and which CGenFF topology/parameter release they belong to. Record it with your simulation.

In [ ]:
if LIG_DIR is None:
    print("LIG_DIR is None: set it in check A.")
else:
    for name in ("lig.rtf", "lig.prm"):
        if (Path(LIG_DIR) / name).is_file():
            program, files = read_cgenff_version(Path(LIG_DIR) / name)
            print(f"{name}: CGenFF program version {program}; topology/parameter files version {files}")

### Check B: stereocentres in the SDF sent to CHARMM-GUI

Set `SDF_PATH` to the SDF you uploaded. RDKit assigns CIP labels from the 3D coordinates; every backbone stereocentre should be R.

In [ ]:
from rdkit import Chem

SDF_PATH = None  # e.g. repo_root / "examples/output/polymer_structures/PHB4_R.sdf"

if SDF_PATH is None:
    print("SDF_PATH is None: set it to the SDF uploaded to Ligand Reader & Modeler.")
else:
    mol = Chem.MolFromMolFile(str(SDF_PATH), removeHs=False)
    Chem.AssignStereochemistryFrom3D(mol)
    centres = Chem.FindMolChiralCenters(mol, includeUnassigned=True, useLegacyImplementation=False)
    for index, label in centres:
        print(f"atom {index:>4} {mol.GetAtomWithIdx(index).GetSymbol()}: {label}")
    print(f"{len(centres)} stereocentres; all R: {bool(centres) and all(label == 'R' for _, label in centres)}")

## Files to keep

Keep the whole Ligand Reader & Modeler download (`lig/`, `toppar/`, `gromacs/`, `ligandrm.*`) together with the exact SDF you uploaded. Treat them as one matched set: atom names, charges and parameters in these files belong together.

## Next step

Steps 2–5 (Solution Builder with `lig_g.rtf` + `lig.prm`, water box, ions, GROMACS input, and the HPC run folder) are in `06B_cgenff_gromacs_pha_enzyme_in_water.ipynb`.

## Limitations

- CGenFF targets drug-like small molecules; PHA oligomer parameters are assigned by analogy.
- There is no PHA-specific validation of these parameters. Review penalties and, for production work, consider targeted validation of the highest-penalty terms.